# 확산 모델 실습

**Diffusion Model · DDPM**

데이터에 잡음을 점차 더하는 과정을 역으로 학습해 잡음에서 표본을 생성하는 모델.

소재 분야에서 이해하기: 잡음에서 시작해 결정 구조 후보를 생성한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [확산 모델 원논문](https://arxiv.org/abs/2006.11239)

## 1. 잡음을 더하는 과정 정의

1차원 데이터로 확산 모델(DDPM)의 순방향·역방향을 모두 구현합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 목표 분포: 두 개의 봉우리(예: 두 종류의 안정상)
target = np.concatenate([rng.normal(-1.5, 0.25, 3000), rng.normal(1.5, 0.25, 3000)])
steps = 40
betas = np.linspace(1e-3, 0.25, steps)
alphas = np.cumprod(1 - betas)
print('마지막 단계의 신호 잔존 비율 %.4f (거의 순수 잡음)' % alphas[-1])
plt.hist(target, bins=80, density=True); plt.xlabel('x'); plt.title('target distribution'); plt.show()

In [ ]:
# 순방향: x_t = sqrt(alpha_t) x_0 + sqrt(1-alpha_t) * noise
t_index = rng.integers(0, steps, target.size)
noise = rng.normal(0, 1, target.size)
x_t = np.sqrt(alphas[t_index]) * target + np.sqrt(1 - alphas[t_index]) * noise

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for axis, step in zip(axes, [0, 10, 25, 39]):
    sample = np.sqrt(alphas[step]) * target + np.sqrt(1 - alphas[step]) * rng.normal(0, 1, target.size)
    axis.hist(sample, bins=60, density=True); axis.set_title('t = %d' % step)
plt.tight_layout(); plt.show()

## 2. 잡음을 예측하는 모델 학습

In [ ]:
from sklearn.neural_network import MLPRegressor

inputs = np.column_stack([x_t, t_index / steps])
denoiser = MLPRegressor(hidden_layer_sizes=(128, 128), max_iter=600, random_state=0).fit(inputs, noise)
print('잡음 예측 오차(표준편차 1 기준) %.3f' % np.mean(np.abs(denoiser.predict(inputs) - noise)))

## 3. 역방향 샘플링

순수 잡음에서 출발해 단계마다 잡음을 덜어냅니다.

In [ ]:
def sample(n=4000, seed=3):
    """표준 DDPM 역방향 한 단계: x_{t-1} = (x_t - beta/sqrt(1-alpha_bar) * eps) / sqrt(alpha) + sigma z"""
    local = np.random.default_rng(seed)
    x = local.normal(0, 1, n)
    for step in range(steps - 1, -1, -1):
        predicted = denoiser.predict(np.column_stack([x, np.full(n, step / steps)]))
        alpha = 1 - betas[step]
        alpha_bar = alphas[step]
        mean = (x - betas[step] / np.sqrt(1 - alpha_bar) * predicted) / np.sqrt(alpha)
        x = mean if step == 0 else mean + np.sqrt(betas[step]) * local.normal(0, 1, n)
        x = np.clip(x, -6, 6)
    return x

generated = sample()
plt.hist(target, bins=80, density=True, alpha=0.5, label='target')
plt.hist(generated, bins=80, density=True, alpha=0.5, label='generated')
plt.legend(); plt.xlabel('x'); plt.show()
for label, mask in [('-1.5 근처', np.abs(generated + 1.5) < 0.5), ('+1.5 근처', np.abs(generated - 1.5) < 0.5),
                    ('중앙(0 근처)', np.abs(generated) < 0.5)]:
    print('%-12s 표본 비율 %.3f' % (label, mask.mean()))
print('두 봉우리에 표본이 모이고 중앙이 비어 있으면 역방향 과정이 분포를 학습한 것입니다.')

## 4. 해석

확산 모델은 "잡음을 조금씩 걷어내는 방법"만 배우면 잡음에서 데이터를 만들 수 있습니다.
결정 구조 생성 모델도 원자 좌표에 같은 원리를 적용합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#diffusion-model)을 여세요.